In [1]:
import pandas as pd
import numpy as np

In [2]:
class Orderblock:
    def __init__(self, data, capital = 1000000):
        self.data = data
        self.position = 0
        self.trade_count = 0
        self.current_trade = 0
        self.trade_created_at = 0
        self.previous_balance = capital
        self.balance = capital
        self.capital = capital
        self.unit = 0
        self.sl_amount = 0
        self.tp_amount = 0
        self.entry = 0
        self.profit = 0
        self.loss = 0
        self.equity = []
        self.leverage = 100
        self.profit_returns = []
        self.loss_returns = []

    def analyze_gold_obs(self, displacement_mult=2.0, forward_window=10):
        """
        df: DataFrame with ['o', 'h', 'l', 'c', 'ATR_14']
        type: either bullish or bearish
        displacement_mult: How much stronger the move must be than the OB candle to count.
        forward_window: How many candles to look ahead for return after a hit.
        """
        obs = []
        active_zones = []

        for i in range(1, len(self.data) - 1):


            if self.current_trade != 1:
                if self.position == 1:
                    self.current_trade = 1
                    self.buy_position(i, amount=self.balance)

            # TP
            # profit = pricechange x lot size x 100
            # (unit x current price)/ leverage ratio = balance needed
            if self.data['h'].iloc[i] >= self.entry+ (0.02 * self.entry) and self.position == 1:
                print('tp hit')
                self.sell(i, unit=self.unit)
                self.current_trade = 0
                self.position = 0
                self.tp_amount +=1
                # self.profit += 1
                self.previous_balance = self.balance
                self.equity.append({
                    'time': self.data.index[i],
                    'balance': round(self.balance)
                })

            # SL
            if self.data['l'].iloc[i] <= self.entry - (0.005 * self.entry) and self.position == 1:
                print('sl hit')
                self.sell(i, unit=self.unit)
                self.current_trade = 0
                self.position = 0
                self.sl_amount += 1
                # self.loss += 1
                self.previous_balance = self.balance
                self.equity.append({
                    'time': self.data.index[i],
                    'balance': round(self.balance)
                })

            curr = self.data.iloc[i]
            prev = self.data.iloc[i - 1]

            # --- 1. IDENTIFY NEW ORDER BLOCKS ---
            # Bullish OB: Last Bearish candle before a strong Bullish move
            if curr['c'] > curr['o'] and (curr['c'] - curr['o']) > (prev['h'] - prev['l']) * displacement_mult and \
                    prev['body'] < prev['ATR_14']:
                if prev['c'] < prev['o']:
                    active_zones.append({
                        'type': 'Bullish',
                        'top': prev['h'],
                        'bottom': prev['l'],
                        'created_at': i,
                        'created_time': self.data['time'].iloc[i],
                        'status': 'Active'
                    })

            for zone in active_zones:
                if zone['status'] != 'Active': continue

                # Check for INVALIDATION (Body Close through zone)
                if zone['type'] == 'Bullish' and curr['c'] < zone['bottom']:
                    zone['status'] = 'Invalidated'
                    continue
                hit = False
                if zone['type'] == 'Bullish':
                    # Low crosses mid of order block
                    if curr['l'] <= ((zone['top'] + zone['bottom'])/2) and i - zone['created_at'] > 10:
                        hit = True
                if hit:
                    # Capture the Return
                    future_idx = min(i + forward_window, len(self.data) - 1)
                    future_price = self.data.iloc[future_idx]['c']
                    ret = ((future_price - ((zone['top'] + zone['bottom'])/2))/((zone['top'] + zone['bottom'])/2)) if zone['type'] == 'Bullish' else (curr['c'] - future_price)


                    print('going long')
                    self.position = 1
                    self.trade_created_at = i
                    self.entry = (zone['top'] + zone['bottom'])/2


                    obs.append({
                        'Type': zone['type'],
                        'Created_At': self.data.index[zone['created_at']],
                        'time': self.data['time'].iloc[i],
                        'vol_regime': self.data['vol_regime'].iloc[i],
                        'sessions': self.data['sessions'].iloc[i],
                        'Hit_At': self.data.index[i],
                        # 'highest_after_hit':self.data['highest'].iloc[i],
                        # 'lowest_after_hit':self.data['lowest'].iloc[i],
                        'Return': ret,
                        'Zone_Top': zone['top'],
                        'Zone_Bottom': zone['bottom']
                    })
                    zone['status'] = 'Mitigated'  # Mark as done

            if i - self.trade_created_at > 20 and self.position == 1:
                print('going neutral')
                self.sell(i, unit=self.unit)
                self.current_trade = 0
                self.position = 0
                if self.balance > self.previous_balance:
                    self.profit += 1
                    forward_return = ( self.data['c'].iloc[i] - self.entry)/self.entry
                    self.profit_returns.append(forward_return)
                elif self.balance < self.previous_balance:
                    self.loss += 1
                    forward_return = ( self.data['c'].iloc[i] - self.entry)/self.entry
                    self.loss_returns.append(forward_return)

                self.previous_balance = self.balance
                self.equity.append({
                    'time': self.data.index[i],
                    'balance': round(self.balance)
                })

        # return pd.DataFrame(obs)

    def show_data(self):
        return self.data

    def get_values(self, bar):
        date = str(self.data.index[bar])
        price = round(self.data['c'].iloc[bar], 5)
        return  date, price

    def buy_position(self, bar, amount = None, unit = None):
        date, price = self.get_values(bar)
        # unit = (-1% of balance/(-0.5% of entry * 100))100
        if amount is not None:
            unit = int(0.01 * amount/((0.005 * self.entry)* 100) * 100)
            # unit = int(amount / self.entry)

        spread_cost = unit * self.data['spread'].iloc[self.trade_created_at]
        self.balance -= spread_cost
        self.balance -= unit * self.entry
        self.unit += unit
        self.trade_count += 1

        print(75 * "-")
        print(self.balance)
        print("{} | +++ Buy POSITION +++".format(date))
        print("{} |  Buying {} for {}".format(date, unit, round(price, 5)))
        perf = (self.balance - self.capital) / self.capital * 100
        # print("{} | net performance (%) = {}".format(date, round(perf, 2) ))
        print("{} | number of trades executed = {}".format(date, self.trade_count))
        print(75 * "-")

    def sell(self, bar, amount = None, unit = None):
        date, price = self.get_values(bar)
        if amount is not None:
            unit = int(amount / price)
        self.balance += unit * price
        self.unit -= unit
        self.trade_count += 1

        print(75 * "-")
        print(self.balance)
        print("{} | +++ Neutral POSITION +++".format(date))
        print("{} |  selling {} for {}".format(date, unit, round(price, 5)))
        perf = (self.balance - self.capital) / self.capital * 100
        print("{} | net performance (%) = {}".format(date, round(perf, 2) ))
        print("{} | number of trades executed = {}".format(date, self.trade_count))
        print(75 * "-")